# Stage 1A — Preprocessing and exploratory QC of the accepted aligned counts

**Input:** immutable, sample-ID-aligned raw count archive, four cohorts D1–D4. This notebook executes read-only scientific checks of the **9,983 shared gene symbols**, per-sample `log2(1+CPM)` transformation, and the **train-only** prevalence/variance filtering and scaling used in Stage 5.

The original dataset correction in Stage 0A **does not** normalize expression. This stage is a distinct, later modeling transformation. No held-out cohort outcomes are used to fit a filter/scaler. See `12_cohort_characterization.ipynb` for existing PCA, BioProject and global cohort figures; this notebook does not duplicate those models.

In [ ]:
from pathlib import Path
import os, sys, json
from datetime import datetime, timezone
import pandas as pd

candidates = [os.environ.get("CMPS355_ROOT"), Path.cwd(), Path.cwd().parent,
              "/content/drive/MyDrive/CMPS355_Research"]
ROOT = next((Path(x).resolve() for x in candidates if x and
             (Path(x)/"data/validated/ibd_dataset_sample_aligned.zip").is_file() and
             (Path(x)/"src/cmps355").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set CMPS355_ROOT to the project root containing data/validated and src/cmps355")
sys.path.insert(0,str(ROOT / "src"))
RUN_ID = os.environ.get("CMPS355_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("PROJECT_ROOT:", ROOT)
print("RUN_ID:", RUN_ID)


In [ ]:
from cmps355.early_preprocessing_qc import run
RUN_DIR=ROOT/'runs/early_preprocessing'/RUN_ID
report=run(ROOT,RUN_DIR)
assert report['status']=='PASS_PREPROCESSING_EXPLORATION_AND_TRAIN_ONLY_FILTER'
print('EARLY_PREPROCESSING_STATUS:',report['status'])
print('DATASET_SHA:',report['dataset_sha256'])

In [ ]:
display(pd.read_csv(RUN_DIR/'cohort_exploration.csv'))
display(pd.read_csv(RUN_DIR/'loco_training_preprocessing.csv'))

In [ ]:
from IPython.display import Image,display
for name in ['figure_raw_shared_gene_library_sizes.png',
             'figure_logcpm_distributions.png','figure_train_only_filter.png']:
    print(name)
    display(Image(filename=str(RUN_DIR/name),width=760))

## Interpretation and leakage boundary

- All four original gene count tables are **raw integer counts** (D1–D3 have 10,061 gene columns; D4 has 41,257); their all-cohort shared-gene intersection contains **9,983 gene names** before inspecting labels.
- CPM denominator here is **the per-sample count sum over those 9,983 shared genes**, not a sequencing-run library-size estimate from all mapped reads. This matches the Stage 5 Python baseline and Stage 10 TENTACLES downstream classifier, **not** the native TENTACLES edgeR normalization.
- Stage 5 feature eligibility: CPM > 1 in at least 10% of **outer training samples**, nonzero training variance. StandardScaler fits to **training samples only**. A separate inner-cohort training fold recomputes both operations for tuning.
- Exploratory plots may use pooled unlabeled data but are **never reused as a supervised train-time transformation**. The PCA and class/batch exploration from Stage 12 remain preserved separately.
- No batch correction was applied across the four held-out cohorts. Cohort/source confounding and tissue heterogeneity should be discussed in the manuscript.